**Assign labels for each category**

In [ ]:
#import os for file handling
import os
#import numpy library for mathematical calculations
import numpy as np

#declare the data path
data_path = r"train_data_2"

#list out all the files inside the data_path
labels = os.listdir(data_path)
print(labels)

#generate three different labels in numbers
categories = np.arange(len(labels))
print(categories)

#generate the dictionary of labels and categories
category_dict = {k: int(v) for k,v in zip(labels,categories)}
print(category_dict)


['Barack Obama', 'Donald Trump', 'George W Bush']
[0 1 2]
{'Barack Obama': 0, 'Donald Trump': 1, 'George W Bush': 2}


**Extract the face of the image using haarcascade_frontalface_default.xml**

In [ ]:
#import cv2 for image handling
import cv2
import os

#load the cascadeClassifier which can detect faces using edge/featuer detection
face_classifier = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

#create empty list to store data:cropped face images
data = []

#create an empty list to store targets:corresponding numeric labels
target = []

#first loop through each person's folder
for label in labels:
    #build the full path to the current person's image folder
    # e.g., "train_data_2/Barack Obama"
    person_path = os.path.join(data_path,label)
    
    #list out all the image names inside each person's folder
    img_names = os.listdir(person_path)
    #print(img_path)
    
    #loop through each image file in the current person's folder
    for img_name in img_names:
        #build a freah path for every individual image
        current_img_path = os.path.join(person_path,img_name)
        #print(img_path)
        
        #read the image directly as grayscale
        gray_img = cv2.imread(current_img_path,0)
        
        #skip if the image couldn't be read properly
        if gray_img is None:
            print(f"Could not read image: {current_img_path}")
            continue
            
        #detect faces in the grayscale image using the cascadeClassifier
        #returns a list of rectangles (x,y,w,h) where faces are detected
        faces = face_classifier.detectMultiScale(gray_img)

        try:
            #loop through each detected face bounding box coordinates
            #x,y--> top left corner of the face rectangle
            #w.h--> width and height of the rectangle
            for(x,y,w,h) in faces:
                #crop the detecte face part from the grayscaled image
                cropped_face = gray_img[y:y+h,x:x+w]
                
                #Diaplay the cropped face in a window to visually verify the detected face
                cv2.imshow("Cropped Face",cropped_face)
                
                #wait indefinitely for a key press from the user 
                #the key code is stored in "k" for decion making
                k = cv2.waitKey(0)
                #check if the user pressed "y"(ASCII code 121) and approved the face to add into data list
                if k==121:
                    #uniform size is requiered for consistent model training 
                    #therefore resize the approved image to a fixed size of 50x50  pixels
                    resized_face = cv2.resize(cropped_face, (50,50))
                    #append the approved and resized image into data list
                    data.append(resized_face)
                    #append corresponding numeric label into target list
                    # category_dict maps name -> number (e.g., 'Barack Obama' -> 0)
                    target.append(category_dict[label])
                else:
                    pass
        except Exception as e:
            print(f"Error on image {current_img_path}:{e}")
            
#close all openCV display windows after processing is complete
cv2.destroyAllWindows()
        

**Flattening:convert 2D data array into 1D array**

In [3]:
#import numpy library
import numpy as np

#convert the lists into numy arrays
data = np.array(data)
target = np.array(target)

#print the shape of data array
print('Before Flattening:',data.shape)

#flatten the images:this converts 50x50 2D array into 1D array of 2500 numbers
data = data.reshape(data.shape[0], data.shape[1]*data.shape[2])
print('After Flattening:',data.shape)

#save .npy files for later use
np.save('data',data)
np.save('target',target)

Before Flattening: (333, 50, 50)
After Flattening: (333, 2500)


**print count of each person in the data list after approving the cropped faces**

In [4]:
#print count of data for each person
import collections
print(collections.Counter(target))

Counter({np.int64(1): 127, np.int64(2): 114, np.int64(0): 92})


**Load the saved .npy files for trainig the model**

In [5]:
import numpy as np
data = np.load('data.npy')
target = np.load('target.npy')

In [6]:
#import suppor vector machine to identify the face (classification tasks)
from sklearn.svm import SVC
#import Principal Component Analysis to reduce the features from 2500 to small set
from sklearn.decomposition import PCA
#import make_pipeline to chain PCA and SVC into single workflow
#pipeline ensures that PCA reduce the features before train or predict on it by SVM
from sklearn.pipeline import make_pipeline

#configure PCA by: 
    #reducing the features into 150
    #normalizing each coponent to have eqaul variance
    #fixing randomness 
pca = PCA(n_components=150, whiten=True, random_state=42)

#Configure SVC by:
#set kernel to 'rbf'.it is ideal for complex patterns like facial recognition.
svc = SVC(kernel='rbf')

#combine PCA and SVC 
model = make_pipeline(pca,svc)

**Split the data for train & test.**

In [28]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
train_data,test_data,train_target,test_target = train_test_split(data,target,test_size=0.1,random_state=42)

# Normalize pixel values to 0-1 range
train_data = train_data / 255.0
test_data = test_data / 255.0 

#print dataset summary
print('Total Sampels:',{len(data)})
print('Total Trainig Samples:',{len(train_data)})
print('Testing Samples:',{len(test_data)})
print('Feature Size',{train_data.shape[1]})
print('-----------------------------------------------------------------')

# Use 5-fold cross-validation on the full dataset to get a robust accuracy estimate
# The pipeline (PCA + SVC) is evaluated end-to-end on each fold
scores = cross_val_score(model, data/255.0, target, cv=5)

# scores is an array of 5 accuracy values, one per fold
print(f"Cross-validation scores: {scores}")

# Mean gives the overall average accuracy across all 5 folds
print(f"Mean accuracy: {scores.mean():.2f}")

# Std shows consistency - lower std means more stable performance across folds
print(f"Standard deviation: {scores.std():.2f}")

#train the model on the fixed reproducible split
model.fit(train_data,train_target)

#get the predictions on test data
predicted_target = model.predict(test_data)

Total Sampels: {333}
Total Trainig Samples: {299}
Testing Samples: {34}
Feature Size {2500}
-----------------------------------------------------------------
Cross-validation scores: [0.70149254 0.89552239 0.89552239 0.78787879 0.86363636]
Mean accuracy: 0.83
Standard deviation: 0.07


### 🔍 Hyperparameter Tuning with GridSearchCV

Systematically evaluate combinations of PCA dimensions and SVM hyperparameters across 5-fold cross-validation to discover the optimal configuration and prevent overfitting.

In [30]:
# Import GridSearchCV for systematic hyperparameter tuning
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Define the parameter grid across both pipeline stages (PCA & SVC)
param_grid = {
    "pca__n_components": [50, 100, 150],
    "svc__C": [1, 5, 10, 50],
    "svc__gamma": ["scale", 0.001, 0.005]
}

# Configure 5-fold cross-validated grid search across candidate parameters
grid = GridSearchCV(model, param_grid, cv=5, n_jobs=-1, verbose=1)
grid.fit(train_data, train_target)

print(f"Best Hyperparameters: {grid.best_params_}")
print(f"Best 5-Fold Cross-Validation Accuracy: {grid.best_score_:.4f}")

# Evaluate the optimized estimator
best_model = grid.best_estimator_
best_predictions = best_model.predict(test_data)
print(f"Optimized Model Test Accuracy: {accuracy_score(test_target, best_predictions):.4f}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Hyperparameters: {'pca__n_components': 50, 'svc__C': 1, 'svc__gamma': 'scale'}
Best 5-Fold Cross-Validation Accuracy: 0.8364
Optimized Model Test Accuracy: 0.8387


**Calculate the accuracy of the model and generate the evaluvation report**

In [29]:
#check the accuracy score on test & predicted targets
from sklearn.metrics import accuracy_score
acc = accuracy_score(test_target,predicted_target)
print("Acuracy score of the model:",acc)
print('----------------Classification Report------------------------')

#generate the classification report
from sklearn.metrics import classification_report
report = classification_report(test_target,predicted_target,target_names=['Barack Obama', 'Donald Trump', 'George W Bush'])
print(report)
print('----------------Confusion Matrix-----------------------------')

#generate the confusion matrix
from sklearn.metrics import confusion_matrix
matrix = confusion_matrix(test_target,predicted_target)
print(matrix)

Acuracy score of the model: 0.8529411764705882
----------------Classification Report------------------------
               precision    recall  f1-score   support

 Barack Obama       1.00      0.71      0.83         7
 Donald Trump       0.92      0.86      0.89        14
George W Bush       0.75      0.92      0.83        13

     accuracy                           0.85        34
    macro avg       0.89      0.83      0.85        34
 weighted avg       0.87      0.85      0.85        34

----------------Confusion Matrix-----------------------------
[[ 5  0  2]
 [ 0 12  2]
 [ 0  1 12]]


**Visualize the confusion matrix with heatmap.**

In [ ]:
#seaborn library is a data visulization library that built on top of the matplotlib.
import seaborn as sns
import matplotlib.pyplot as plt

#configure the  heatmap:
    #matrix: a 3x3 grid of numbers showing how well the model predicted each person.
    #annot=True: shows numbers inside boxes
    #fmt='d': Tells seaborn to display numbers as integers
    #xticklabels,yticklabels : Labels for the x and y axis
sns.heatmap(matrix, annot=True, fmt='d', xticklabels=['Obama', 'Trump', 'Bush'], yticklabels=['Obama', 'Trump', 'Bush'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

**Save the model for later use.**

In [ ]:
import joblib
joblib.dump(model, 'SVM-Face Recognition.sav')